# SB3 monitor episode boundary regression

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import unittest,tempfile,json
from pathlib import Path
import numpy as np
from environment_adapter import BaselineState
from baseline_rewards import TaskReward,MaxSupportReward
from types import SimpleNamespace
import gymnasium as gym
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from telemetry_recording import Recorder
class AdapterFixture(gym.Env):
    observation_space=gym.spaces.Box(-1,1,(2,),dtype=np.float32)
    action_space=gym.spaces.Discrete(2)
    state=SimpleNamespace(episode=0)
    def reset(self,**kwargs):
        self.state=SimpleNamespace(episode=self.state.episode+1);self.count=0
        return np.zeros(2,dtype=np.float32),{'episode':self.state.episode}
    def step(self,action):
        self.count+=1;end=self.count==2
        info={'episode':self.state.episode,'decision':self.count,'raw_score_signal':1.,
              'b':float(self.count==1),'fresh':False,'episode_end_reason':'fixture' if end else None}
        return np.zeros(2,dtype=np.float32),info['b'],end,False,info

class Checks(unittest.TestCase):
    def test_monitor_summary_and_raw_episode_id_survive_vectorization(self):
        with tempfile.TemporaryDirectory() as directory:
            recorder=Recorder(AdapterFixture(),directory)
            vec=DummyVecEnv([lambda:Monitor(recorder)])
            try:
                vec.reset()
                _,_,done,infos=vec.step([0])
                self.assertFalse(done[0]);self.assertNotIn('episode',infos[0])
                self.assertEqual(infos[0]['episode_id'],1)
                _,_,done,infos=vec.step([0])
                self.assertTrue(done[0]);self.assertEqual(infos[0]['episode_id'],1)
                self.assertEqual(infos[0]['episode']['r'],1.)
                self.assertEqual(infos[0]['episode']['l'],2)
            finally:
                recorder.finish();vec.close()
            events=[json.loads(line) for line in (Path(directory)/'events.jsonl').read_text().splitlines()]
            self.assertEqual([r['episode'] for r in events if r['event']=='decision'],[1,1])
print('SB3 monitor episode boundary regression definitions/execution completed.')


Frozen increasing-preference scoring definitions/execution completed.
Task progress and fresh-pair rewards definitions/execution completed.
Matched observation action and window adapter definitions/execution completed.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Frozen runtime contract definitions/execution completed.
Decision and episode telemetry definitions/execution completed.
SB3 monitor episode boundary regression definitions/execution completed.


In [3]:
suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print('Tests run:',result.testsRun)

test_monitor_summary_and_raw_episode_id_survive_vectorization (__main__.Checks) ... 

ok


----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


Tests run: 1
